# 17 — Explanation selective risk policy

**Objective.** Build the frozen 2009 operational explanation-loss panel, instantiate the finite candidate-family LTT/SCRC application, calibrate accepted-set explanation/prediction risks, evaluate chronological transport on 2010, and run IID theorem-behaviour simulations.

**Scientific contract.** All 2009 losses are deterministic bounded functionals of objects frozen through 2008. Missing or failed explanations receive loss 1. Any guarantee is aggregate, finite-family, assumption-dependent, conditional on frozen pre-calibration objects, and withheld unless the exact theorem/procedure has been independently checked. The chronological track never receives a distribution-free claim.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("17", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json, joblib, os, yaml
import numpy as np
import pandas as pd
from cruxvc.esrc import build_operational_explanation_panel
from cruxvc.explanations import GowerKNNDonorSampler, active_feature_groups, load_feature_groups
from cruxvc.io import read_json, read_table, write_json, write_table
from cruxvc.manifest import signing_key_from_environment, verify_protocol_lock
from cruxvc.selective import apply_gate, build_frozen_candidate_family, calibrate_finite_policy_family
from cruxvc.synthetic import simulate_iid_policy_violations
from cruxvc.workflow import fit_refit_calibrated_model

signing_key = signing_key_from_environment()
require_hmac = bool(CFG["execution"]["require_hmac_for_final_test"])
verify_protocol_lock(P.locks / "design_lock.json", signing_key=signing_key, require_hmac=require_hmac)
design = read_json(P.protocol / "design_snapshot.json")

candidates = read_table(P.selective / "frozen_candidate_policies.parquet")
scores2009 = read_table(P.selective / "risk_calibration_gate_scores_2009.parquet")
test_scores = read_table(P.selective / "test_gate_scores_common_population.parquet")
test_losses = read_table(P.selective / "test_losses_common_population.parquet")
features = read_table(P.processed / "features_strict.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
splits = read_table(P.protocol / "split_ids.parquet")
registry = read_table(P.models / "calibration_registry.parquet")
task_registry = read_table(P.protocol / "final_task_registry.parquet")
seed_registry = pd.read_csv(P.protocol / "seed_registry.csv")
background_registry = pd.read_csv(P.protocol / "explanation_background_ids.csv")
statistical = yaml.safe_load((P.config / "statistical_analysis.yaml").read_text(encoding="utf-8"))

frame = features.merge(cohort, on=["case_id", "company_permalink", "t0"]).merge(
    splits[["case_id", "time_block"]], on="case_id"
)
development = frame[frame["time_block"].astype(str).eq("development")].copy()
calibration2008 = frame[frame["time_block"].astype(str).eq("probability_calibration")].copy()
risk2009 = frame[frame["time_block"].astype(str).eq("risk_calibration")].copy()
feature_columns = [c for c in features.columns if c not in {"case_id", "company_permalink", "t0"}]
groups = active_feature_groups(load_feature_groups(P.config / "feature_groups.yaml"), feature_columns)
risk_ids = scores2009["case_id"]
if set(risk_ids) != set(risk2009["case_id"]):
    raise RuntimeError("The 2009 gate-score IDs do not match the frozen risk-calibration block")
risk2009 = risk2009.set_index("case_id").reindex(risk_ids).reset_index()
risk_labels = risk2009.set_index("case_id")
score_columns = sorted(set(candidates["score_name"]).intersection(scores2009.columns))
if set(candidates["score_name"]) - set(score_columns):
    missing = sorted(set(candidates["score_name"]) - set(score_columns))
    raise RuntimeError(f"Frozen ESRC candidate scores are missing from 2009 inputs: {missing}")

In [ ]:
# Build the finite frozen 2009 operational panel unless an identical completed
# panel already exists. CRUX_ESRC_SKIP_FULL_PANEL=1 is an explicit diagnostic
# escape hatch; its proxy losses can never support an ESRC risk-control claim.
full_panel_path = P.selective / "risk_calibration_explanation_losses_2009.parquet"
panel_attr_path = P.attributions / "esrc_risk_calibration_attributions_2009.parquet"
panel_faithfulness_path = P.controls / "esrc_risk_calibration_faithfulness_2009.parquet"
panel_audit_path = P.audits / "17_esrc_operational_panel_audit.json"
panel_generated_outputs = []
skip_full_panel = os.environ.get("CRUX_ESRC_SKIP_FULL_PANEL", "0").strip() == "1"

if not full_panel_path.exists() and not skip_full_panel:
    prediction_outcome = str(design["esrc_operational_panel"]["prediction_outcome"])
    primary = (
        registry[
            registry["analysis_role"].eq("matched_reference")
            & registry["outcome"].eq(prediction_outcome)
        ]
        .sort_values(["platt_oof_log_loss", "family", "config_id"])
        .iloc[0]
    )
    matched_rows = registry[
        registry["analysis_role"].eq("matched_reference")
        & registry["family"].eq(primary["family"])
        & registry["config_id"].eq(primary["config_id"])
        & registry["outcome"].isin(CFG["outcomes"]["confirmatory"])
    ].copy()
    if set(matched_rows["outcome"]) != set(CFG["outcomes"]["confirmatory"]):
        raise RuntimeError("The frozen matched configuration is unavailable for every confirmatory outcome")
    deployment_models = {
        str(row.outcome): joblib.load(row.calibrated_model_path)
        for row in matched_rows.itertuples(index=False)
    }

    panel_n = int(design["esrc_operational_panel"]["reference_panel_refits"])
    panel_tasks = (
        task_registry[
            task_registry["task_type"].eq("bootstrap")
            & task_registry["outcome"].eq(prediction_outcome)
            & task_registry["family"].eq(primary["family"])
            & task_registry["config_id"].eq(primary["config_id"])
        ]
        .sort_values(["refit_slot", "task_id"])
        .head(panel_n)
    )
    if len(panel_tasks) != panel_n:
        raise RuntimeError(f"Frozen ESRC panel requires {panel_n} refits; found {len(panel_tasks)}")
    reference_models = []
    for task in panel_tasks.itertuples(index=False):
        reference_models.append((
            str(task.task_id),
            fit_refit_calibrated_model(
                development,
                calibration2008,
                feature_columns=feature_columns,
                outcome=prediction_outcome,
                family=str(task.family),
                parameters=json.loads(task.parameters_json),
                seed=int(task.seed),
            ),
        ))

    first_background_id = sorted(background_registry["background_id"].unique())[0]
    background_ids = (
        background_registry[background_registry["background_id"].eq(first_background_id)]
        .sort_values("order")["case_id"]
        .head(int(design["explanation_budgets"]["background_n"]))
    )
    background = development[development["case_id"].isin(background_ids)].copy()
    if len(background) != len(background_ids):
        raise RuntimeError("Frozen ESRC background IDs are incomplete in development data")
    approximation_seed = int(
        seed_registry[seed_registry["purpose"].eq("approximation")]
        .sort_values("index")["seed"]
        .iloc[0]
    )
    sampler = GowerKNNDonorSampler.fit(
        development,
        feature_columns,
        k=int(statistical["faithfulness"]["conditional_sampler_primary_k"]),
    )
    panel = build_operational_explanation_panel(
        deployment_models=deployment_models,
        reference_models=reference_models,
        cases=risk2009,
        background=background,
        groups=groups,
        feature_columns=feature_columns,
        sampler=sampler,
        faithfulness_scale=float(design["esrc_operational_panel"]["faithfulness_scale"]),
        approximation_seed=approximation_seed,
        n_orderings=int(design["esrc_operational_panel"]["permutation_orderings"]),
        faithfulness_steps=tuple(statistical["faithfulness"]["deletion_steps"]),
        faithfulness_random_repetitions=int(design["esrc_operational_panel"]["faithfulness_random_repetitions"]),
        seed=int(CFG["execution"]["random_seed"]) + 171,
        prediction_outcome=prediction_outcome,
    )
    write_table(panel.losses, full_panel_path)
    write_table(panel.attributions, panel_attr_path)
    write_table(panel.faithfulness, panel_faithfulness_path)
    write_json({
        **panel.audit,
        "primary_family": str(primary["family"]),
        "primary_config_id": str(primary["config_id"]),
        "reference_task_ids": panel_tasks["task_id"].tolist(),
        "background_id": first_background_id,
        "losses_are_deterministic_bounded_functionals": True,
        "missing_or_failed_explanations_receive_loss_one": True,
    }, panel_audit_path)
    panel_generated_outputs = [full_panel_path, panel_attr_path, panel_faithfulness_path, panel_audit_path]

if full_panel_path.exists():
    required_loss_columns = {"case_id", "L_S", "L_C", "L_F", "L_Y"}
    risk_losses = read_table(full_panel_path)
    missing = required_loss_columns - set(risk_losses.columns)
    if missing:
        raise RuntimeError(f"Full 2009 explanation-loss panel lacks columns: {sorted(missing)}")
    risk_losses = risk_losses.set_index("case_id").reindex(risk_ids).reset_index()
    if risk_losses[["L_S", "L_C", "L_F", "L_Y"]].isna().any().any():
        raise RuntimeError("Full 2009 explanation-loss panel is incomplete for the frozen risk-calibration IDs")
    if ((risk_losses[["L_S", "L_C", "L_F", "L_Y"]] < 0) | (risk_losses[["L_S", "L_C", "L_F", "L_Y"]] > 1)).any().any():
        raise RuntimeError("Every ESRC calibration loss must be bounded in [0, 1]")
    panel_audit = read_json(panel_audit_path) if panel_audit_path.exists() else {"failed_components": ["missing_panel_audit"]}
    full_panel_available = True
    panel_quality_pass = (
        not panel_audit.get("failed_components")
        and int(risk_losses.get("reference_panel_available", pd.Series(0)).min())
            == int(risk_losses.get("reference_panel_expected", pd.Series(1)).max())
        and int(risk_losses.get("construct_contrasts_available", pd.Series(0)).min())
            == int(risk_losses.get("construct_contrasts_expected", pd.Series(1)).max())
    )
    loss_source = "frozen_operational_explanation_panel"
else:
    proxy_columns = [c for c in ["predicted_explanation_loss", "epistemic_variance", "density_ood"] if c in scores2009.columns]
    proxy = scores2009[proxy_columns].copy()
    for column in proxy:
        lo, hi = proxy[column].quantile([0.01, 0.99])
        proxy[column] = ((proxy[column] - lo) / max(float(hi - lo), 1e-12)).clip(0, 1)
    predicted = scores2009["predicted_class"].to_numpy(dtype=int)
    y = risk_labels[str(CFG["selective_risk"]["prediction_outcome"])].to_numpy(dtype=int)
    ones = pd.Series(np.ones(len(proxy)), index=proxy.index)
    risk_losses = pd.DataFrame({
        "case_id": risk_ids.to_numpy(),
        "L_S": proxy.get("epistemic_variance", ones).to_numpy(),
        "L_C": proxy.get("predicted_explanation_loss", ones).to_numpy(),
        "L_F": proxy.get("density_ood", ones).to_numpy(),
        "L_Y": (predicted != y).astype(float),
    })
    full_panel_available = False
    panel_quality_pass = False
    panel_audit = {"skipped": True, "reason": "CRUX_ESRC_SKIP_FULL_PANEL=1 or panel unavailable"}
    loss_source = "diagnostic_proxy_not_operational_loss"
calibration = scores2009[["case_id", *score_columns]].merge(
    risk_losses[["case_id", "L_S", "L_C", "L_F", "L_Y"]],
    on="case_id",
    how="inner",
    validate="one_to_one",
)

In [ ]:
budgets = {
    "L_S": float(CFG["selective_risk"]["rho_S"]),
    "L_C": float(CFG["selective_risk"]["rho_C"]),
    "L_F": float(CFG["selective_risk"]["rho_F"]),
    "L_Y": float(CFG["selective_risk"]["rho_Y"]),
}
policy_results, selected = calibrate_finite_policy_family(
    calibration,
    candidates,
    score_columns=score_columns,
    loss_budgets=budgets,
    q_min=float(CFG["selective_risk"]["q_min"]),
    familywise_delta=float(CFG["selective_risk"]["familywise_delta"]),
    minimum_accepted=int(CFG["selective_risk"]["accepted_calibration_min"]),
)
theorem_independently_checked = os.environ.get("CRUX_ESRC_THEOREM_CHECKED", "0").strip() == "1"
policy_results["loss_source"] = loss_source
policy_results["full_2009_explanation_panel_available"] = full_panel_available
policy_results["panel_quality_pass"] = panel_quality_pass
policy_results["theorem_independently_checked"] = theorem_independently_checked
policy_results["eligible_for_guarantee_claim"] = (
    policy_results["certified"]
    & full_panel_available
    & panel_quality_pass
    & theorem_independently_checked
)

In [ ]:
temporal_rows = []
loss_name_map = {
    "L_S": "explanation_instability",
    "L_C": "construct_fragility",
    "L_F": "faithfulness_proxy_loss",
    "L_Y": "prediction_error",
}
if selected is not None:
    gate = str(selected["score_name"])
    accepted = apply_gate(
        test_scores[gate],
        float(selected["threshold"]),
        lower_is_better=bool(selected["lower_is_better"]),
    )
    for loss_name, test_column in loss_name_map.items():
        empirical = float(test_losses.loc[accepted, test_column].mean()) if accepted.any() else np.nan
        temporal_rows.append({
            "loss": loss_name,
            "empirical_2010_risk": empirical,
            "budget": budgets[loss_name],
            "violates_budget": bool(np.isfinite(empirical) and empirical > budgets[loss_name]),
            "policy_loss_source": loss_source,
            "chronological_guarantee_claim": False,
        })
    temporal_coverage = float(accepted.mean())
else:
    temporal_coverage = 0.0
temporal = pd.DataFrame(temporal_rows)


In [ ]:
def _sigmoid(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    return 1 / (1 + np.exp(-values))


def _iid_scores_and_losses(x, y):
    # The score is deployment-time computable. Four bounded losses have both
    # shared and distinct signal, so simultaneous control is nontrivial.
    predicted_probability = _sigmoid(1.5 * x[:, 0] + 0.6 * x[:, 1])
    uncertainty = 1 - np.abs(2 * predicted_probability - 1)
    scores = pd.DataFrame({
        "uncertainty": uncertainty,
        "epistemic_proxy": _sigmoid(0.9 * x[:, 2] + 0.5 * uncertainty - 1.0),
        "density_proxy": np.clip(np.linalg.norm(x, axis=1) / 5.0, 0, 1),
    })
    losses = pd.DataFrame({
        "L_S": _sigmoid(4.0 * uncertainty + 0.35 * x[:, 2] - 3.0),
        "L_C": _sigmoid(3.5 * uncertainty + 0.30 * np.abs(x[:, 3]) - 2.7),
        "L_F": _sigmoid(2.2 * uncertainty + 0.25 * x[:, 2] - 1.4),
        "L_Y": ((predicted_probability >= 0.5).astype(int) != np.asarray(y, dtype=int)).astype(float),
    })
    return scores, losses


def iid_builder(cal_x, cal_y, pop_x, pop_y, repetition):
    split = len(cal_x) // 2
    development_scores, _ = _iid_scores_and_losses(cal_x[:split], cal_y[:split])
    risk_scores, risk_losses = _iid_scores_and_losses(cal_x[split:], cal_y[split:])
    population_scores, population_losses = _iid_scores_and_losses(pop_x, pop_y)
    simulated_candidates = build_frozen_candidate_family(
        development_scores,
        CFG["selective_risk"]["candidate_acceptance_grid"],
    )
    risk_calibration = pd.concat([risk_scores.reset_index(drop=True), risk_losses.reset_index(drop=True)], axis=1)
    calibrated, selected_simulated = calibrate_finite_policy_family(
        risk_calibration,
        simulated_candidates,
        score_columns=list(development_scores.columns),
        loss_budgets=budgets,
        q_min=float(CFG["selective_risk"]["q_min"]),
        familywise_delta=float(CFG["selective_risk"]["familywise_delta"]),
        minimum_accepted=int(CFG["selective_risk"]["accepted_calibration_min"]),
    )
    if selected_simulated is None:
        return {
            "selected": False,
            "any_constraint_violation": False,
            "population_coverage": 0.0,
            **{f"population_{name}": np.nan for name in budgets},
        }
    gate = str(selected_simulated["score_name"])
    accepted = apply_gate(
        population_scores[gate],
        float(selected_simulated["threshold"]),
        lower_is_better=bool(selected_simulated["lower_is_better"]),
    )
    population_risks = {
        name: float(population_losses.loc[accepted, name].mean()) if accepted.any() else 1.0
        for name in budgets
    }
    coverage = float(accepted.mean())
    violation = coverage < float(CFG["selective_risk"]["q_min"]) or any(
        population_risks[name] > budgets[name] for name in budgets
    )
    return {
        "selected": True,
        "selected_score": gate,
        "population_coverage": coverage,
        "any_constraint_violation": bool(violation),
        **{f"population_{name}": value for name, value in population_risks.items()},
    }


iid = simulate_iid_policy_violations(
    iid_builder,
    repetitions=int(PROFILE["iid_policy_repetitions"]),
    calibration_n=2000,
    population_n=20000 if PROFILE["name"] == "smoke" else 100000,
    delta=float(CFG["selective_risk"]["familywise_delta"]),
    seed=int(CFG["execution"]["random_seed"]) + 170,
)


In [ ]:
policy_path = write_table(policy_results, P.selective / "esrc_policy_calibration_2009.csv")
temporal_path = write_table(temporal, P.selective / "esrc_temporal_transport_2010.csv")
iid_path = write_table(iid.repetitions, P.selective / "esrc_iid_simulation_repetitions.parquet")
eligible_for_guarantee = bool(
    selected is not None
    and full_panel_available
    and panel_quality_pass
    and theorem_independently_checked
    and bool(policy_results.loc[policy_results["selected"], "eligible_for_guarantee_claim"].all())
)
if eligible_for_guarantee:
    method_status = "applied_extension_eligible_for_assumption_qualified_claim"
elif full_panel_available and panel_quality_pass:
    method_status = "operational_panel_evaluated_but_theorem_check_pending_not_certifiable"
elif full_panel_available:
    method_status = "operational_panel_contains_forced_loss_failures_not_certifiable"
else:
    method_status = "diagnostic_proxy_only_not_certifiable"
summary_path = write_json({
    **iid.summary,
    "selected_policy": selected.to_dict() if selected is not None else None,
    "temporal_coverage": temporal_coverage,
    "loss_source": loss_source,
    "full_2009_explanation_panel_available": full_panel_available,
    "panel_quality_pass": panel_quality_pass,
    "panel_audit": panel_audit,
    "theorem_independently_checked": theorem_independently_checked,
    "eligible_for_guarantee_claim": eligible_for_guarantee,
    "method_status": method_status,
    "chronological_track_guarantee": False,
}, P.selective / "esrc_summary.json")
CTX.recorder.complete([
    *panel_generated_outputs,
    policy_path,
    temporal_path,
    iid_path,
    summary_path,
])
print(read_json(summary_path))